In [55]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

In [56]:
df=pd.read_csv("train.txt",sep=";",header=None,names=['text','emotion'])

In [57]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [58]:

df.isnull().sum()

text       0
emotion    0
dtype: int64

In [59]:
unique_emotions = df['emotion'].unique()

emotion_numbers = {}

for i, emo in enumerate(unique_emotions):
    emotion_numbers[emo] = i

df['emotion'] = df['emotion'].map(emotion_numbers)

print(emotion_numbers)
print(df['emotion'].value_counts())

{'sadness': 0, 'anger': 1, 'love': 2, 'surprise': 3, 'fear': 4, 'joy': 5}
emotion
5    5362
0    4666
1    2159
4    1937
2    1304
3     572
Name: count, dtype: int64


In [60]:
df.shape


(16000, 2)

In [61]:
df['text']=df['text'].apply(lambda x :x.lower())

In [62]:
import string

In [63]:
def remove_punc(txt):
    return txt.translate(str.maketrans('', '', string.punctuation))

In [64]:
df['text']=df['text'].apply(remove_punc)

In [65]:

def remove_numbers(txt):
    return txt.translate(str.maketrans('', '', string.digits))

In [66]:
df['text']=df['text'].apply(remove_numbers)

In [67]:
import re

def clean_text(txt):
    txt = re.sub(r'https?://\S+|www\.\S+', '', txt)  # remove URLs
    txt = re.sub(r'\d+', '', txt)                    # remove numbers
    return txt

In [68]:
df['text']=df['text'].apply(clean_text)

In [69]:
import emoji

def remove_emojis(txt):
    return emoji.replace_emoji(txt, replace='')

In [70]:
pip install emoji

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [71]:
df['text']=df['text'].apply(remove_emojis)

In [72]:
import nltk
nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\rajes\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [73]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [74]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words('english'))

def remove_stopwords(txt):
    return ' '.join(
        word for word in txt.split()
        if word.lower() not in stop_words
    )

In [75]:
df['text']=df['text'].apply(remove_stopwords)

In [76]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [77]:
from sklearn.feature_extraction.text import CountVectorizer

# Documents
documents = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is great"
]

# Create CountVectorizer
vectorizer = CountVectorizer(ngram_range=(2,2))

# Learn vocabulary and transform documents into BoW matrix
X = vectorizer.fit_transform(documents)

# Display vocabulary
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

# Display Bag-of-Words matrix
print("\nBoW Matrix:")
print(X.toarray())

Vocabulary:
['is great' 'is the' 'love pasta' 'love pizza' 'pasta is' 'pizza is'
 'the best']

BoW Matrix:
[[0 0 0 1 0 0 0]
 [0 1 0 0 0 1 1]
 [0 0 1 0 0 0 0]
 [1 0 0 0 1 0 0]]


In [78]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Documents
documents = [
    "I love pizza",
    "Pizza is the best",
    "I love pasta",
    "Pasta is great"
]

# Create CountVectorizer
vectorizer = TfidfVectorizer()

# Learn vocabulary and transform documents into BoW matrix
X = vectorizer.fit_transform(documents)

# Display vocabulary
print("Vocabulary:")
print(vectorizer.get_feature_names_out())

# Display Bag-of-Words matrix
print("\nBoW Matrix:")
print(X.toarray())

Vocabulary:
['best' 'great' 'is' 'love' 'pasta' 'pizza' 'the']

BoW Matrix:
[[0.         0.         0.         0.70710678 0.         0.70710678
  0.        ]
 [0.55528266 0.         0.43779123 0.         0.         0.43779123
  0.55528266]
 [0.         0.         0.         0.70710678 0.70710678 0.
  0.        ]
 [0.         0.66767854 0.52640543 0.         0.52640543 0.
  0.        ]]


In [79]:
X=df['text']
y=df['emotion']

In [80]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.20, random_state=42
)


In [81]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer

In [82]:
bow_vectoriser=CountVectorizer()


In [83]:
X_train_bow=bow_vectoriser.fit_transform(X_train)

In [84]:
X_test_bow = bow_vectoriser.transform(X_test)

In [85]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

In [86]:
nb_model=MultinomialNB()

In [87]:
nb_model.fit(X_train_bow,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13361)","[[1.,1.,0.,...,0.,0.,1.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,0.,...,0.,1.,0.], [0.,0.,0.,...,0.,0.,0.], [1.,0.,0.,...,0.,0.,0.], [0.,0.,1.,...,1.,2.,0.]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13361)","[[-10.07,-10.07,-10.76,...,-10.76,-10.76,-10.07], [ -9.6 ,-10.29,-10.29,...,-10.29,-10.29,-10.29], [-10.06,-10.06,-10.06,...,-10.06, -9.36,-10.06], [ -9.79, -9.79, -9.79,..., -9.79, -9.79, -9.79], [ -9.53,-10.22,-10.22,...,-10.22,-10.22,-10.22], [-10.91,-10.91,-10.22,...,-10.22, -9.81,-10.91]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13361


In [88]:
prediction=nb_model.predict(X_test_bow)

In [89]:
accuracy_score(y_test,prediction)

0.768125

In [90]:
tfidf_vectoriser=TfidfVectorizer()
X_train_tfidf=tfidf_vectoriser.fit_transform(X_train)
X_test_tfidf = tfidf_vectoriser.transform(X_test)
tfidf_model=MultinomialNB()

In [91]:
tfidf_model.fit(X_train_tfidf,y_train)

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](6,)","[3720.,1732.,1008., 459.,1540.,4341.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](6,)","[-1.24,-2. ,-2.54,-3.33,-2.12,-1.08]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](6,)","[0,1,2,3,4,5]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](6, 13361)","[[0.5 ,0.48,0. ,...,0. ,0. ,0.31], [0.48,0. ,0. ,...,0. ,0. ,0. ], [0. ,0. ,0. ,...,0. ,0.29,0. ], [0. ,0. ,0. ,...,0. ,0. ,0. ], [0.36,0. ,0. ,...,0. ,0. ,0. ], [0. ,0. ,0.3 ,...,0.26,0.7 ,0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](6, 13361)","[[ -9.65, -9.67,-10.05,...,-10.05,-10.05, -9.79], [ -9.41, -9.8 , -9.8 ,..., -9.8 , -9.8 , -9.8 ], [ -9.69, -9.69, -9.69,..., -9.69, -9.44, -9.69], [ -9.59, -9.59, -9.59,..., -9.59, -9.59, -9.59], [ -9.47, -9.77, -9.77,..., -9.77, -9.77, -9.77], [-10.14,-10.14, -9.88,..., -9.91, -9.61,-10.14]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,13361


In [92]:
prediction_tfidf=nb_model.predict(X_test_tfidf)

In [95]:
accuracy_score(y_test,prediction_tfidf)

0.7275

In [96]:
from sklearn.linear_model import LogisticRegression

# Create model
lr_model = LogisticRegression(max_iter=1000)

# Train
lr_model.fit(X_train_tfidf, y_train)

# Predict
prediction = lr_model.predict(X_test_tfidf)

print(prediction)

[0 5 0 ... 5 5 0]


In [ ]:
accuracy_score(y_test,prediction_tfidf)